# Day 15 - Reading features/v1/ from inside SageMaker

Run this **on the SageMaker notebook instance**, kernel `conda_python3`.

Notice what's missing compared to `01_s3_io.ipynb`: no `profile="ds"`, no keys, no `storage_options`.
The notebook instance runs *as* its execution role, the same way the EC2 box did on Day 6.

In [ ]:
# The conda_python3 kernel doesn't ship pyarrow. Install, then Kernel -> Restart Kernel.
# Re-run after every instance restart: only /home/ec2-user/SageMaker survives a stop/start.
%pip install -q pyarrow

In [ ]:
import boto3
import sagemaker

sess = sagemaker.Session()
role = sagemaker.get_execution_role()

print("Region :", sess.boto_region_name)
print("Role   :", role)
print("Caller :", boto3.client("sts").get_caller_identity()["Arn"])

## 1. How big is it, before loading anything?

`ml.t3.medium` has 4 GB of RAM. The full feature table is ~7.2M rows x 21 columns - roughly 1.5 GB in pandas,
more while it's being built. Check first, load second. This reads only Parquet footers.

In [ ]:
import pyarrow.dataset as ds

FEATURES = "s3://dave-ds-lab-ca/features/v1/"

dataset = ds.dataset(FEATURES, format="parquet", partitioning="hive")
print(dataset.schema)
print()
for split in ["train", "valid"]:
    n = dataset.count_rows(filter=ds.field("split") == split)
    print(f"{split:6s} {n:>12,} rows")

## 2. Load the validation split only

The `filter` is pushed down to the partition folders: only `split=valid/` is downloaded.
Same idea as Day 10's partition pruning, but in Python instead of Athena.

In [ ]:
import time

t0 = time.perf_counter()
valid = dataset.to_table(filter=ds.field("split") == "valid").to_pandas()
print(f"{len(valid):,} rows in {time.perf_counter() - t0:.1f}s, "
      f"{valid.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory")
valid.head()

In [ ]:
valid.describe().T[["mean", "min", "max"]].round(2)

## 3. Prove the role's limits

The role can read `features/` and write `models/` and `outputs/`. It cannot write `raw/`.
Expect the second call to fail with AccessDenied.

In [ ]:
s3 = boto3.client("s3")

s3.put_object(Bucket="dave-ds-lab-ca", Key="outputs/day15/hello.txt", Body=b"written from SageMaker")
print("outputs/ write: OK")

try:
    s3.put_object(Bucket="dave-ds-lab-ca", Key="raw/should-fail.txt", Body=b"nope")
    print("raw/ write: OK  <- the policy is too broad, check the role")
except s3.exceptions.ClientError as e:
    print("raw/ write:", e.response["Error"]["Code"])

## 4. Free the memory, then STOP THE INSTANCE

SageMaker console -> Notebook instances -> `ds-lab-notebook` -> Actions -> **Stop**.
It bills every hour it is `InService`, whether a kernel is running or not.

In [ ]:
del valid
import gc; gc.collect()
print("done - now stop the instance")